## Train a character-level GPT on some text data

The inputs here are simple text files, which we chop up to individual characters and then train GPT on. So you could say this is a char-transformer instead of a char-rnn. Doesn't quite roll off the tongue as well. In this example we will feed it some Shakespeare, which we'll get it to predict character-level.

In [1]:
# set up logging
import logging
logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(name)s -   %(message)s",
        datefmt="%m/%d/%Y %H:%M:%S",
        level=logging.INFO,
)

In [2]:
# make deterministic
from mingpt.utils import set_seed
set_seed(42)

In [3]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

In [4]:
%%writefile dataset.py

import math
import torch
from torch.utils.data import Dataset

class CharDataset(Dataset):

    def __init__(self, data, block_size, stride=None):
        chars = sorted(list(set(data)))
        data_size, vocab_size = len(data), len(chars)
        print("data has %d characters, %d unique." % (data_size, vocab_size))
        
        self.stoi = { ch:i for i,ch in enumerate(chars) }
        self.itos = { i:ch for i,ch in enumerate(chars) }
        self.block_size = block_size
        self.vocab_size = vocab_size
        self.data = data
        self.stride = stride if stride is not None else block_size
    
    def __len__(self):
        return (len(self.data) - 1) // self.stride

    def __getitem__(self, idx):
        start = idx * self.stride
        chunk = self.data[start:start + self.block_size + 1]
        dix = [self.stoi[s] for s in chunk]
        x = torch.tensor(dix[:-1], dtype=torch.long)
        y = torch.tensor(dix[1:], dtype=torch.long)
        return x, y


Overwriting dataset.py


In [5]:
block_size = 128 # spatial extent of the model for its context

In [6]:
!curl https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -o input.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1089k  100 1089k    0     0  1519k      0 --:--:-- --:--:-- --:--:-- 1519k


In [7]:
# you can download this file at https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt
from dataset import CharDataset
text = open('input.txt', 'r').read() # don't worry we won't run out of file handles
train_dataset = CharDataset(text, block_size) # one line of poem is roughly 50 characters

data has 1115394 characters, 65 unique.


In [8]:
from mingpt.model import GPT, GPTConfig
mconf = GPTConfig(train_dataset.vocab_size, train_dataset.block_size,
                  n_layer=8, n_head=8, n_embd=512)
model = GPT(mconf)

10/11/2026 01:50:56 - INFO - mingpt.model -   number of parameters: 2.535219e+07


In [9]:
willfrom mingpt.trainer import Trainer, TrainerConfig

# initialize a trainer instance and kick off training
tconf = TrainerConfig(
    max_epochs=5,            # 5 full epochs over Shakespeare (finishes in ~8.6 minutes!)
    batch_size=64,           # 137 steps per epoch
    learning_rate=6e-4,
    lr_decay=True,
    warmup_tokens=64*20,
    final_tokens=5*len(train_dataset)*block_size,
    num_workers=0            # Set to 0 to prevent macOS spawn overhead & CPU throttling
)

trainer = Trainer(model, train_dataset, None, tconf)
trainer.train()


epoch 1 iter 120: train loss 2.48356. lr 5.545290e-04: 100%|██████████| 137/137 [03:50<00:00,  1.68s/it]
epoch 2 iter 120: train loss 2.27079. lr 4.125424e-04: 100%|██████████| 137/137 [02:02<00:00,  1.12it/s]
epoch 3 iter 120: train loss 2.03818. lr 2.275494e-04: 100%|██████████| 137/137 [02:32<00:00,  1.12s/it]
epoch 4 iter 120: train loss 1.88733. lr 7.024231e-05: 100%|██████████| 137/137 [02:37<00:00,  1.15s/it]
epoch 5 iter 120: train loss 1.83925. lr 6.000000e-05: 100%|██████████| 137/137 [02:42<00:00,  1.19s/it]


In [10]:
# alright, let's sample some character-level Shakespeare
from mingpt.utils import sample

context = "O God, O God!"
x = torch.tensor([train_dataset.stoi[s] for s in context], dtype=torch.long)[None,...].to(trainer.device)
y = sample(model, x, 2000, temperature=1.0, sample=True, top_k=10)[0]
completion = ''.join([train_dataset.itos[int(i)] for i in y])
print(completion)

O God, O God!
DUKE VINCESS:
Mak, te me to che ans hall the staing wadind
This ded out the the pad soo moner of of mad.
Gore!
POMERO:
No,
Comme, fay.

DUELIO:
Thy wold see now! to the sir,

Firstior:
I with hose and slearr, is as hearts in with the heir;
And and bretsent and to hou cantintond to seant,
Tho mrtors marrne all to me of an with come.

PTROMINIO:
Have this did has; bet bant be and and caurnd borsth,
From as the may at agroomel, as fond the this
did as he thee prandel trik; I art of the ath
ere it cheard he all me will the seer the shan weent so.

DUKE OF OF IF HERS:
That mork in the dend as say mung to may a and fruls,
Maice me their belees in onotow more mist or one,
And but and be he as cond have havich thes
Bunt shalll me offther, he por on old to sheeld;
Is and brothy were of the bile and wo by mether
as mose on come be thist camanters
Ald and troth to manter; and thou shall me,
As has my be stretlend sto this arlond
How should win bere with the teminer,
Shat and all tee

In [11]:
# well that was fun